# 61. `CPBackgroundSpec` and `CPBackgroundCategory`

**Objectives:**

- Add a charge-aware combinatorial background to a `CPFitSession` with
  `CPBackgroundSpec` (a single shared shape via `resolved_minus_shape`).
- Build the same background directly as a `CPBackgroundCategory` fed to
  `CPJointNLL`, and confirm it reproduces the same NLL.
- Read CLAUDE.md's "CP fits share one normalization across charges" section: both
  constructions normalize the background jointly over `(Dalitz, charge)`, not as two
  independent per-charge shapes.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV²,
daughter indices start at zero.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede any numerical work: amplitudes use complex128.

import jax.numpy as jnp
import numpy as np

from dalitzplotfitter import (
    CPBackgroundCategory, CPBackgroundSpec, CPFitSession, CPJointNLL, CPRealImag,
    DecayChannel, DecayModel, NonResonant, Parameter, RealImag, Resonance,
    generate_cp_toy,
)

## 1. Two charge amplitudes from shared parameters

A single K* resonance with a charge-dependent non-resonant term
(`CPRealImag`, `c_q = (x + q*dx) + i*(y + q*dy)`), as in
[tutorial 6](tutorial_06_joint_cp_fit.ipynb).

In [2]:
cp = CPRealImag(*[
    Parameter.coefficient(f"NR.{name}", value, owner="NR", bounds=bounds, step=0.02)
    for name, value, bounds in [
        ("x", 0.8, (-2, 2)), ("y", 0.3, (-2, 2)),
        ("dx", 0.10, (-0.4, 0.4)), ("dy", -0.08, (-0.4, 0.4)),
    ]
])

def make_model(parent, daughters, charge):
    return DecayModel(
        DecayChannel(parent, daughters),
        [Resonance("Kstar", (0, 2), RealImag(1, 0), mass=0.8958, width=0.0474, spin=1),
         NonResonant(cp.for_charge(charge), name="NR")],
        normalization_method="square-dalitz", normalization_pair=(0, 2),
        normalization_resolution=70,
    )

plus_model = make_model("B+", ("K+", "pi+", "pi-"), +1)
minus_model = make_model("B-", ("K-", "pi-", "pi+"), -1)
truth = {p.name: p.value for p in plus_model.parameters}
plus_data, minus_data = generate_cp_toy(
    plus_model, minus_model, 3000, parameters=truth, seed=61,
    inverse_resolution=384, include_momenta=False,
)
print("Charge counts:", plus_data.size, minus_data.size)

Charge counts: 1611 1389


## 2. High-level: `CPBackgroundSpec` inside `CPFitSession`

One flat shape shared by both charges: leaving `minus_shape=None` makes
`resolved_minus_shape` fall back to `plus_shape`, per the class docstring.

In [3]:
def background_shape(events):
    return jnp.ones_like(events["s12"])

fraction = Parameter("signal_fraction", 0.8, bounds=(0.05, 0.99), step=0.02)
session = CPFitSession(
    plus_model, minus_model, plus_data, minus_data,
    signal_fraction=fraction,
    backgrounds=(CPBackgroundSpec("comb", plus_shape=background_shape),),
)
nll_high_level = float(session.objective(truth))
print(f"CPFitSession + CPBackgroundSpec: NLL(truth) = {nll_high_level:.6f}")

CPFitSession + CPBackgroundSpec: NLL(truth) = 14210.830394


## 3. Low-level: `CPBackgroundCategory` + `CPJointNLL`

By hand, this is exactly what `CPFitSession` did in step 2 (its own
`_build_background`): evaluate the shape on each charge's data sample for
`plus_values`/`minus_values`, then reduce it on each charge's own normalization
sample with `mean(weights * shape)` for `plus_normalization`/`minus_normalization`.
`CPBackgroundCategory.normalization` is the **combined** `plus_normalization +
minus_normalization` -- the joint-charge normalization CLAUDE.md describes -- so the
background's `plus_density`/`minus_density` (`values / normalization`) are each
divided by the *sum* across both charges, exactly like the signal density
`S_q = eps_q |A_q|^2 / (I_+ + I_-)`.

In [4]:
plus_cache = plus_model.prepare_cache(plus_data, plus_model.normalization_sample)
minus_cache = minus_model.prepare_cache(minus_data, minus_model.normalization_sample)

plus_norm_sample = plus_model.normalization_sample
minus_norm_sample = minus_model.normalization_sample

category = CPBackgroundCategory(
    "comb",
    plus_values=background_shape(plus_data.as_dict()),
    minus_values=background_shape(minus_data.as_dict()),
    plus_normalization=jnp.mean(
        plus_norm_sample.weights * background_shape(plus_norm_sample.as_dict())
    ),
    minus_normalization=jnp.mean(
        minus_norm_sample.weights * background_shape(minus_norm_sample.as_dict())
    ),
)

low_level_nll = CPJointNLL(
    plus_cache, minus_cache,
    background_categories=(category,),
    signal_fraction=fraction,
)
nll_low_level = float(low_level_nll(truth))
print(f"CPBackgroundCategory + CPJointNLL: NLL(truth) = {nll_low_level:.6f}")

np.testing.assert_allclose(nll_low_level, nll_high_level, rtol=1e-10)
print("Both constructions give the identical NLL at the same parameter point.")

CPBackgroundCategory + CPJointNLL: NLL(truth) = 14210.830394
Both constructions give the identical NLL at the same parameter point.


## Continue learning

See CLAUDE.md's "CP fits share one normalization across charges" section and
[`docs/cp_coefficients.md`](../../docs/cp_coefficients.md) for why B+/B- share one
normalization (sensitivity to the integrated charge asymmetry), and
[`docs/backgrounds_and_vetoes.md`](../../docs/backgrounds_and_vetoes.md) for multiple
CP background categories and extended-mode yields.

Return to [the course guide](TUTORIALS.md).